# Materi 5: Web Structure Mining — Teori Graf dan Karakteristik Jaringan

**Mata Kuliah:** Penambangan & Pencarian Web (PPW)  
**Program Studi:** Teknik Informatika, Universitas Trunojoyo Madura  
**Materi Perkuliahan:** *Web Structure Mining* (Slide 1 hingga slide *Contoh: Diameter*)

---

### 📌 Garis Besar Materi:
1. **Pendahuluan Web Structure Mining** & Mengapa Graf?
2. **Terminologi Graf vs Jaringan** (Padanan Istilah Lintas Disiplin)
3. **Definisi Formal Graf**: Graf Berarah (*Directed*) & Tak Berarah (*Undirected*)
4. **Representasi Graf**: Matriks Ketetanggaan (*Adjacency Matrix*) & Graf Berbobot (*Weighted*)
5. **Graf Bipartit (*Bigraph*)** dan Atribut Simpul/Sisi (*Attributes*)
6. **Derajat Simpul (*Degree*)**: *In-Degree*, *Out-Degree*, Simpul *Source* dan *Sink*
7. **Distribusi Derajat (*Degree Distribution*)** & Derajat Rata-rata (*Average Degree*)
8. **Kepadatan Graf (*Graph Density*)** & Maknanya pada Jaringan Nyata
9. **Koefisien Klasterisasi (*Clustering Coefficient*)**: Lokal & Rata-rata Global
10. **Lintasan (*Path*)** dan **Jalur Terpendek / Jarak Geodesik (*Shortest Path*)**
11. **Jarak Rata-rata** dan **Diameter Jaringan (*Network Diameter*)** (Studi Kasus Slide 24)

---

## 1. Pendahuluan Web Structure Mining

Dalam lanskap penambangan web (*Web Mining*), terdapat tiga pilar utama:
1. **Web Content Mining**: Menganalisis konten teks, multimedia, dan informasi di dalam halaman web (seperti TF-IDF, Klasifikasi Teks, dan Word Embedding pada materi sebelumnya).
2. **Web Structure Mining**: Mengeksplorasi dan menganalisis pola hubungan serta struktur keterhubungan (topologi) antar dokumen web atau entitas digital.
3. **Web Usage Mining**: Mengidentifikasi pola perilaku pengguna berdasarkan data log web dan navigasi.

### Mengapa World Wide Web (WWW) Dimodelkan sebagai Graf?
Struktur World Wide Web secara alami merepresentasikan graf raksasa:
- **Simpul (*Vertices / Nodes*)**: Merepresentasikan halaman web (URL), dokumen, atau akun pengguna.
- **Sisi / Tepi (*Edges / Links / Arcs*)**: Merepresentasikan hyperlink antar dokumen web, hubungan pertemanan, atau rujukan kutipan.

> **Catatan Terminologi:**
> - **Jaringan (*Network*)**: Istilah yang sering merujuk pada sistem di dunia nyata (contoh: jaringan sosial, jaringan jalan, jaringan web).
> - **Graf (*Graph*)**: Model representasi matematis formal dari sebuah jaringan.
> Dalam literatur informatika, istilah *Network* dan *Graph* sering digunakan secara bergantian (*interchangeable*).

### Padanan Istilah Lintas Disiplin Ilmu
| Disiplin Ilmu | Simpul (*Point*) | Hubungan / Garis (*Line*) |
| :--- | :--- | :--- |
| **Matematika (*Math*)** | Vertices | Edges, Arcs |
| **Ilmu Komputer (*Computer Science*)** | Nodes | Links |
| **Fisika (*Physics*)** | Sites | Bonds |
| **Sosiologi (*Sociology*)** | Actors | Ties, Relations |

In [1]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set style visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Library pendukung Web Structure Mining siap digunakan!")

Library pendukung Web Structure Mining siap digunakan!


## 2. Definisi Formal Graf: Berarah & Tak Berarah

Secara matematis, sebuah graf $G$ didefinisikan sebagai pasangan terurut:
$$\mathcal{G} = (V, E)$$
di mana:
- $V \neq \emptyset$ adalah himpunan simpul (*vertices*).
- $E \subseteq V \times V$ adalah himpunan sisi (*edges*).

Dua simpul $a$ dan $b$ disebut **bertetangga (*adjacent*)** jika terdapat sisi $(a, b) \in E$.

### Perbedaan Graf Tak Berarah dan Graf Berarah:
1. **Graf Tak Berarah (*Undirected Graph*)**:
   - Sisi bersifat simetris: $(a, b) = (b, a)$. Tidak memiliki arah panah.
   - Contoh di dunia nyata: Tautan ko-penulis jurnal (*co-authorship*), pertemanan di Facebook (timbal balik), jaringan jalan dua arah.
2. **Graf Berarah (*Directed Graph / Digraph*)**:
   - Sisi memiliki arah tertentu dan disebut **busur (*arc*)**: $(a, b) \neq (b, a)$.
   - Busur $(a, b)$ berarti hubungan berawal dari $a$ menuju $b$ ($a \to b$).
   - Contoh di dunia nyata: **Hyperlink pada World Wide Web (WWW)**, relasi mengikuti (*following*) di Twitter/X atau Instagram, transaksi keuangan, riwayat panggilan telepon.

In [2]:
# Membuat representasi graf tak berarah dan graf berarah
G_tak_berarah = nx.Graph()
G_tak_berarah.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'C'), ('B', 'D')])

G_berarah = nx.DiGraph()
G_berarah.add_edges_from([('A', 'B'), ('A', 'C'), ('C', 'A'), ('B', 'D'), ('D', 'C')])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Visualisasi Graf Tak Berarah
pos_tb = nx.spring_layout(G_tak_berarah, seed=42)
nx.draw_networkx(G_tak_berarah, pos_tb, ax=axes[0], node_color='lightblue', 
                  node_size=1200, font_size=12, font_weight='bold', edge_color='gray', width=2)
axes[0].set_title("Graf Tak Berarah (Undirected)\nContoh: Ko-penulis / Jaringan Pertemanan", fontsize=11, fontweight='bold')
axes[0].axis('off')

# Visualisasi Graf Berarah
pos_b = nx.spring_layout(G_berarah, seed=42)
nx.draw_networkx(G_berarah, pos_b, ax=axes[1], node_color='lightgreen', 
                  node_size=1200, font_size=12, font_weight='bold', edge_color='gray', 
                  arrows=True, arrowsize=20, arrowstyle='-|>', width=2, connectionstyle="arc3,rad=0.1")
axes[1].set_title("Graf Berarah (Directed)\nContoh: Hyperlink Web / Following Twitter", fontsize=11, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. Matriks Ketetanggaan (*Adjacency Matrix*)

Sebuah graf dapat direpresentasikan secara komputasional dalam bentuk **Matriks Ketetanggaan ($A$)** berukuran $N \times N$, di mana $N$ adalah jumlah simpul:

$$A = [a_{ij}], \quad a_{ij} = \begin{cases} 1, & \text{jika } (i, j) \in E \text{ (ada sisi dari } i \text{ ke } j) \\ 0, & \text{jika tidak ada sisi} \end{cases}$$

### Sifat Penting Matriks Ketetanggaan:
1. **Graf Tak Berarah**: Matriks ketetanggaan selalu **simetris** terhadap diagonal utama ($a_{ij} = a_{ji}$).
2. **Graf Berarah**: Matriks ketetanggaan umumnya **tidak simetris** ($a_{ij} \neq a_{ji}$), karena adanya arah koneksi.

### Graf Berbobot vs Tak Berbobot (*Weighted vs Unweighted*):
- **Graf Tak Berbobot**: Nilai elemen matriks biner ($a_{ij} \in \{0, 1\}$).
- **Graf Berbobot**: Nilai $a_{ij}$ menyatakan nilai numerik riil, seperti jarak antar kota (mil/km), frekuensi interaksi, atau kapasitas aliran data.

In [3]:
# Contoh Graf Berbobot (Jaringan Jalan / Jarak dalam Mil)
G_weighted = nx.Graph()
nodes = [1, 2, 3, 4]
G_weighted.add_nodes_from(nodes)
edges_with_weight = [
    (1, 2, 2.0), 
    (1, 3, 0.5),
    (2, 3, 1.0),
    (2, 4, 4.0)
]
G_weighted.add_weighted_edges_from(edges_with_weight)

# Matriks Ketetanggaan Berbobot
adj_matrix_df = nx.to_pandas_adjacency(G_weighted, nodelist=nodes, weight='weight')
print("=== MATRIKS KETETANGGAAN BERBOBOT (SLIDE 8) ===")
print(adj_matrix_df)

## 4. Graf Bipartit (*Bigraph*) dan Atribut Jaringan

### Graf Bipartit (*Bipartite Graph*)
Graf bipartit adalah graf yang himpunan simpulnya dapat dibagi menjadi dua himpunan saling lepas (*disjoint sets*) $U$ dan $V$ sedemikian rupa sehingga setiap sisi hanya menghubungkan simpul di $U$ ke simpul di $V$. Tidak ada sisi yang menghubungkan simpul di dalam himpunan yang sama ($U \to U$ atau $V \to V$).

**Contoh Aplikasi Graf Bipartit di Web & Data Science:**
- Jaringan Film dan Aktor (Aktor $\leftrightarrow$ Film yang dibintangi)
- Sistem Rekomendasi E-Commerce (Pelanggan $\leftrightarrow$ Produk yang dibeli)
- Jaringan Penyakit dan Gejala Medis (Penyakit $\leftrightarrow$ Gejala yang muncul)
- Media Sosial (Pengguna $\leftrightarrow$ Tag atau Foto di Flickr)

### Atribut Simpul dan Sisi (*Attributes*)
Pada jaringan dunia nyata, simpul dan sisi dapat diperkaya dengan metadata atribut:
- **Atribut Simpul**: Usia pengguna, gender, alamat, lokasi geografis, tipe akun.
- **Atribut Sisi**: Waktu komunikasi, jenis relasi (teman dekat, rekan kerja), urutan preferensi (seperti sosiogram Moreno 1960: preferensi pilihan pertama & kedua).

In [4]:
# Contoh Graf Bipartit (Aktor & Film)
from networkx.algorithms import bipartite

B = nx.Graph()
aktors = ['Reza Rahadian', 'Nicholas Saputra', 'Dian Sastro']
films = ['Habibie & Ainun', 'AADC', 'Sri Asih']

B.add_nodes_from(aktors, bipartite=0)
B.add_nodes_from(films, bipartite=1)
B.add_edges_from([
    ('Reza Rahadian', 'Habibie & Ainun'),
    ('Nicholas Saputra', 'AADC'),
    ('Dian Sastro', 'AADC'),
    ('Reza Rahadian', 'Sri Asih'),
    ('Dian Sastro', 'Sri Asih')
])

print("Apakah graf valid sebagai bipartit?", bipartite.is_bipartite(B))

# Visualisasi Graf Bipartit
pos_bip = dict()
pos_bip.update((n, (1, i)) for i, n in enumerate(aktors))
pos_bip.update((n, (2, i)) for i, n in enumerate(films))

plt.figure(figsize=(9, 5))
nx.draw_networkx_nodes(B, pos_bip, nodelist=aktors, node_color='orange', node_size=1500, label='Aktor (U)')
nx.draw_networkx_nodes(B, pos_bip, nodelist=films, node_color='skyblue', node_size=1500, label='Film (V)')
nx.draw_networkx_edges(B, pos_bip, width=2, edge_color='gray')
nx.draw_networkx_labels(B, pos_bip, font_size=10, font_weight='bold')
plt.title("Visualisasi Graf Bipartit: Aktor vs Film", fontsize=12, fontweight='bold')
plt.legend(scatterpoints=1, loc='upper left')
plt.axis('off')
plt.show()

## 5. Derajat Simpul (*Degree*): In-Degree dan Out-Degree

**Derajat (*Degree*)** simpul $v$ mengukur jumlah sisi yang terhubung atau bertetangga dengan $v$.

### Pada Graf Berarah:
Koneksi dibedakan menjadi dua jenis derajat:
1. **Derajat-Masuk (*In-Degree*)** $C_D^{in}(v_i)$:
   Jumlah sisi berarah yang masuk menuju simpul $v_i$. Dalam matriks ketetanggaan, dihitung dengan menjumlahkan elemen pada **kolom ke-$i$**:
   $$C_D^{in}(v_i) = \sum_{j=1, j \neq i}^{|V|} a_{ji}$$
   *Makna Web:* Menggambarkan **popularitas, prestise, atau otoritas** (banyak halaman lain yang mereferensikan halaman ini).

2. **Derajat-Keluar (*Out-Degree*)** $C_D^{out}(v_i)$:
   Jumlah sisi berarah yang keluar dari simpul $v_i$. Dalam matriks ketetanggaan, dihitung dengan menjumlahkan elemen pada **baris ke-$i$**:
   $$C_D^{out}(v_i) = \sum_{j=1, j \neq i}^{|V|} a_{ij}$$
   *Makna Web:* Menggambarkan **peran sebagai hub / katalog direktori** yang banyak merujuk ke sumber daya lain.

3. **Derajat Total (*Total Degree*)**:
   $$k_v = C_D(v) = C_D^{in}(v) + C_D^{out}(v)$$

### Istilah Simpul Khusus:
- **Source (Sumber)**: Simpul yang hanya memiliki tautan keluar, tanpa ada tautan masuk sama sekali ($k^{in} = 0$ dan $k^{out} > 0$).
- **Sink (Muara)**: Simpul yang hanya menerima tautan masuk, tanpa memiliki tautan keluar ($k^{out} = 0$ dan $k^{in} > 0$).

In [5]:
# Replikasi Studi Kasus Slide 11
# Matriks A berukuran 4x4
A = np.array([
    [0, 0, 0, 1],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 1, 0, 0]
])
nodes_4 = [1, 2, 3, 4]

# Buat DiGraph dari matriks ketetanggaan numpy
G_slide11 = nx.from_numpy_array(A, create_using=nx.DiGraph)
mapping = {i: nodes_4[i] for i in range(len(nodes_4))}
G_slide11 = nx.relabel_nodes(G_slide11, mapping)

# Hitung in-degree dan out-degree
in_degrees = dict(G_slide11.in_degree())
out_degrees = dict(G_slide11.out_degree())

df_deg = pd.DataFrame({
    'Simpul': nodes_4,
    'In-Degree (C_D^in)': [in_degrees[n] for n in nodes_4],
    'Out-Degree (C_D^out)': [out_degrees[n] for n in nodes_4],
    'Total Degree (C_D)': [in_degrees[n] + out_degrees[n] for n in nodes_4]
})

print("=== HASIL PERHITUNGAN DERAJAT SIMPUL (SLIDE 11) ===")
print(df_deg.to_string(index=False))
print(f"\nVerifikasi Simpul v2:")
print(f"- C_D^in(v2)  = {in_degrees[2]}")
print(f"- C_D^out(v2) = {out_degrees[2]}")
print(f"- C_D(v2)     = {in_degrees[2] + out_degrees[2]}")
print(f"\nIdentifikasi Simpul Sink: Simpul 3 (k_in = {in_degrees[3]}, k_out = {out_degrees[3]})")

## 6. Distribusi Derajat & Derajat Rata-rata

### Distribusi Derajat (*Degree Distribution*)
Distribusi derajat merangkum proporsi atau frekuensi kemunculan simpul dengan derajat tertentu di seluruh jaringan:
- $P(k)$: Peluang bahwa sebuah simpul yang dipilih secara acak memiliki derajat tepat sebesar $k$.

### Fenomena *Power-Law Distribution* pada WWW
Pada jaringan acak biasa (model Erdős–Rényi), derajat simpul mengikuti distribusi Poisson / lonceng (sebagian besar simpul memiliki derajat mendekati rata-rata).

Namun pada **jaringan web dunia nyata (World Wide Web)**, distribusi derajat mengikuti **Hukum Pangkat (*Power Law*)**:
$$P(k) \sim k^{-\gamma}, \quad 2 < \gamma < 3$$

Artinya:
- Sebagian sangat besar halaman web hanya memiliki sedikit tautan.
- Namun ada segelintir simpul yang memiliki jutaan tautan masuk/keluar, yang disebut sebagai **Hub / Authority Raksasa** (misalnya Google, Wikipedia, YouTube).
- Fenomena ini dikenal sebagai **Scale-Free Network** (jaringan bebas skala).

### Derajat Rata-rata (*Average Degree*)
Misalkan $N$ adalah jumlah simpul dan $L$ adalah total jumlah sisi:
1. **Graf Tak Berarah**:
   $$\langle k \rangle = \frac{2L}{N}$$
2. **Graf Berarah**:
   $$\langle k^{in} \rangle = \langle k^{out} \rangle = \frac{L}{N}$$

> ⚠️ **Peringatan Penting (Slide 15):**
> Pada jaringan dunia nyata dengan distribusi *power-law*, nilai derajat rata-rata $\langle k \rangle$ bisa **menyesatkan (*misleading*)** karena adanya simpul pencilan (*outlier*) raksasa yang mendistorsi nilai rata-rata aritmatika.

In [6]:
# Simulasi Perbandingan Distribusi Derajat:
# 1. Graf Acak Erdős-Rényi vs 2. Graf Skala Bebas Power-Law Barabási-Albert
n_nodes = 500
G_er = nx.erdos_renyi_graph(n=n_nodes, p=0.04, seed=42)
G_ba = nx.barabasi_albert_graph(n=n_nodes, m=2, seed=42)

deg_er = [d for n, d in G_er.degree()]
deg_ba = [d for n, d in G_ba.degree()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram Graf Acak (Poisson-like)
axes[0].hist(deg_er, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title(f"Graf Acak Erdős-Rényi\n(Rata-rata Derajat: {np.mean(deg_er):.2f})", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Derajat (k)")
axes[0].set_ylabel("Frekuensi Simpul")

# Histogram Graf Power-Law (WWW / Skala Bebas)
axes[1].hist(deg_ba, bins=25, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title(f"Graf Web / Scale-Free Power-Law\n(Rata-rata Derajat: {np.mean(deg_ba):.2f})", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Derajat (k)")
axes[1].set_ylabel("Frekuensi Simpul")

plt.tight_layout()
plt.show()

## 7. Kepadatan Graf (*Graph Density*)

**Kepadatan Graf (*Density*)** mengukur tingkat keterhubungan aktual graf dibandingkan dengan jumlah maksimal koneksi yang secara teoritis dapat terbentuk di antara simpul-simpulnya.

### Jumlah Maksimal Sisi ($L_{maks}$):
- **Graf Berarah**:
  $$L_{maks} = N(N - 1)$$
  *(karena setiap dari $N$ simpul dapat mengarahkan busur ke $(N - 1)$ simpul lainnya)*.

- **Graf Tak Berarah**:
  $$L_{maks} = \frac{N(N - 1)}{2}$$
  *(karena relasi dua arah dihitung satu kali saja)*.

### Rumus Kepadatan (*Density*):
$$D = \frac{L}{L_{maks}}$$

- **Contoh dari Slide 16**:
  Jika dari 12 kemungkinan koneksi maksimum sebuah graf memiliki 11 sisi nyata, maka kepadatan graf tersebut adalah:
  $$D = \frac{11}{12} \approx 0{,}9167 \quad (91{,}67\%)$$.

In [7]:
# Simulasi Perhitungan Density Graf Berarah (Slide 16 & 17)
# 4 Simpul: L_maks = 4 * 3 = 12 kemungkinan busur
DG_dense = nx.DiGraph()
DG_dense.add_nodes_from([1, 2, 3, 4])

# Tambahkan 11 sisi dari 12 kemungkinan
all_possible_edges = [(i, j) for i in [1, 2, 3, 4] for j in [1, 2, 3, 4] if i != j]
edges_11 = all_possible_edges[:11]  # ambil 11 sisi
DG_dense.add_edges_from(edges_11)

N = DG_dense.number_of_nodes()
L = DG_dense.number_of_edges()
L_maks = N * (N - 1)
density_manual = L / L_maks
density_nx = nx.density(DG_dense)

print("=== PERHITUNGAN KEPADATAN GRAF (SLIDE 16) ===")
print(f"- Jumlah Simpul (N)      : {N}")
print(f"- Sisi Maksimal (L_maks) : {L_maks}")
print(f"- Sisi Aktual (L)        : {L}")
print(f"- Density (Manual)       : {density_manual:.4f}")
print(f"- Density (NetworkX)     : {density_nx:.4f}")

## 8. Koefisien Klasterisasi (*Clustering Coefficient*)

**Koefisien Klasterisasi (*Clustering Coefficient*)** mengukur kepadatan koneksi *lokal* di antara tetangga-tetangga suatu simpul. Metrik ini menjawab pertanyaan esensial:
> *"Apakah teman-teman (tetangga) dari simpul $v_i$ juga saling berteman satu sama lain?"*

### Rumus Koefisien Klasterisasi Lokal Simpul $v_i$:
Misalkan simpul $v_i$ memiliki derajat $d_i$ dan himpunan tetangga $N_i$:
$$C_i = \frac{k_i}{\frac{d_i(d_i - 1)}{2}} = \frac{2 k_i}{d_i(d_i - 1)}$$
di mana:
- $d_i$ = Derajat simpul $v_i$ (jumlah tetangga langsung).
- $k_i$ = Jumlah sisi tak berarah yang benar-benar ada di antara sesama tetangga $v_i$.
- $\frac{d_i(d_i - 1)}{2}$ = Jumlah maksimal pasangan koneksi yang mungkin terbentuk di antara sesama tetangga $v_i$.
- Nilai $C_i$ selalu berada dalam interval $[0, 1]$.

### Koefisien Klasterisasi Rata-rata Global ($\bar{C}$):
$$\bar{C} = \frac{1}{N} \sum_{i=1}^N C_i$$

### Replikasi Studi Kasus Slide 19:
Pada graf 9 simpul di Slide 19:
- Tinjau **Simpul 6**:
  - Derajat $d_6 = 4$, dengan tetangga $N_6 = \{4, 5, 7, 8\}$.
  - Di antara tetangga tersebut, terdapat $k_6 = 4$ sisi penghubung, yaitu: $(4,5), (5,7), (5,8), (7,8)$.
  - Maka nilai koefisien klasterisasinya adalah:
    $$C_6 = \frac{4}{\frac{4 \times 3}{2}} = \frac{4}{6} = \frac{2}{3} \approx 0{,}667$$
- Koefisien klasterisasi rata-rata jaringan: $\bar{C} = 0{,}61$.
- Nilai ini jauh lebih tinggi daripada graf acak teoretis ($0{,}19$), yang membuktikan adanya **struktur komunitas** yang kuat.

In [8]:
# Replikasi Persis Graf 9 Simpul (Slide 19)
G_slide19 = nx.Graph()
nodes_9 = list(range(1, 10))
G_slide19.add_nodes_from(nodes_9)

# Daftar sisi sesuai topologi Slide 19
edges_9 = [
    (1, 2), (1, 3), (2, 3), (1, 4), (2, 4),
    (6, 4), (6, 5), (6, 7), (6, 8),
    (4, 5), (5, 7), (5, 8), (7, 8),
    (7, 9), (8, 9)
]
G_slide19.add_edges_from(edges_9)

# Perhitungan Koefisien Klasterisasi
clustering_dict = nx.clustering(G_slide19)
avg_clustering = nx.average_clustering(G_slide19)

df_clustering = pd.DataFrame({
    'Simpul': nodes_9,
    'Derajat (d_i)': [G_slide19.degree(n) for n in nodes_9],
    'Clustering Coeff (C_i)': [round(clustering_dict[n], 4) for n in nodes_9]
})

print("=== ANALISIS KOEFISIEN KLASTERISASI (SLIDE 19) ===")
print(df_clustering.to_string(index=False))
print(f"\n• Koefisien Klasterisasi Simpul 6 (C6) : {clustering_dict[6]:.4f} (atau 2/3)")
print(f"• Koefisien Klasterisasi Rata-rata (C)  : {avg_clustering:.4f}")

## 9. Lintasan (*Path*) & Jarak Geodesik (*Shortest Path*)

### 1. Lintasan (*Path / Walk*)
- Lintasan adalah urutan terurut dari serangkaian simpul di mana setiap simpul bertetangga dengan simpul berikutnya: $P = (v_0, v_1, v_2, \dots, v_n)$.
- Panjang lintasan didefinisikan sebagai **jumlah garis/sisi** yang dilalui ($n$).
- Sebuah lintasan **dapat berpotongan dengan dirinya sendiri** dan melintasi sisi yang sama berulang kali (setiap kali sisi dilewati, dihitung secara terpisah).
- *Contoh Slide 21:* Lintasan $A - B - C - B - C - A - D - E - E - B - A$ memiliki panjang 11 langkah.
- Pada graf berarah, lintasan harus selalu searah dengan panah busur.

### 2. Jalur Terpendek / Jarak Geodesik (*Shortest Path Distance*)
- Jarak $d(u, v)$ antara simpul $u$ dan $v$ didefinisikan sebagai **jumlah sisi pada lintasan terpendek** yang menghubungkan keduanya.
- Jika simpul $u$ dan $v$ tidak terhubung (berada di komponen terpisah), maka jaraknya bernilai tak hingga ($d(u, v) = \infty$).
- Pada graf berarah (*digraph*), jarak umumnya **bersifat asimetris**:
  $$d(u, v) \neq d(v, u)$$

In [9]:
# Analisis Jalur Terpendek pada Graf Berarah (Slide 22)
DG_path = nx.DiGraph()
DG_path.add_edges_from([
    ('A', 'B'),
    ('B', 'C'),
    ('C', 'A'),
    ('C', 'D'),
    ('D', 'B')
])

path_AB = nx.shortest_path(DG_path, source='A', target='B')
dist_AB = nx.shortest_path_length(DG_path, source='A', target='B')

path_BA = nx.shortest_path(DG_path, source='B', target='A')
dist_BA = nx.shortest_path_length(DG_path, source='B', target='A')

print("=== JARAK ASIMETRIS PADA GRAF BERARAH (SLIDE 22) ===")
print(f"- Jalur A -> B : {' -> '.join(path_AB)} (Jarak: {dist_AB})")
print(f"- Jalur B -> A : {' -> '.join(path_BA)} (Jarak: {dist_BA})")

## 10. Jarak Rata-rata dan Diameter Jaringan (Studi Kasus Slide 24)

### Jarak Rata-rata (*Average Shortest Path Length*)
Untuk graf terhubung dengan $N$ simpul, jarak rata-rata $\langle d \rangle$ mengukur seberapa efisien informasi atau lalu lintas dapat berpindah antar dua simpul sembarang:
$$\langle d \rangle = \frac{1}{N(N - 1)} \sum_{i \neq j} d(i, j)$$

### Diameter Jaringan ($d_{maks}$)
**Diameter graf** didefinisikan sebagai jarak maksimum antara sepasang simpul mana pun di seluruh graf:
$$d_{maks} = \max_{u, v \in V} d(u, v)$$

> Dengan kata lain, diameter adalah **"lintasan terpendek yang paling panjang"** (*the longest shortest path*) di antara semua pasangan simpul dalam jaringan.

### Replikasi Studi Kasus Slide 24 (*Batas Akhir Materi*):
Pada graf 10 simpul (simpul 0 sampai 9):
- Terdapat pasangan simpul 7 dan 5 yang memiliki jarak terpendek paling jauh.
- Lintasan geodesik terpanjang adalah:
  $$7 \to 2 \to 6 \to 3 \to 4 \to 5$$
- Jumlah sisi sepanjang lintasan ini adalah 5, sehingga **Diameter Graf = 5**.

In [10]:
# Replikasi Studi Kasus Diameter (Slide 24)
G_slide24 = nx.Graph()
nodes_10 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
G_slide24.add_nodes_from(nodes_10)

# Topologi Sisi Graf Slide 24
edges_diameter = [
    (7, 2),  # Langkah 1
    (2, 6),  # Langkah 2
    (6, 3),  # Langkah 3
    (3, 4),  # Langkah 4
    (4, 5),  # Langkah 5 (Lintasan Diameter 7-2-6-3-4-5)
    (7, 8), (2, 8), (6, 8),
    (6, 1), (3, 1),
    (4, 0), (5, 0)
]
G_slide24.add_edges_from(edges_diameter)

# Perhitungan Metrik Jaringan
diameter_val = nx.diameter(G_slide24)
avg_path_length = nx.average_shortest_path_length(G_slide24)
lintasan_terpanjang = nx.shortest_path(G_slide24, source=7, target=5)

print("=== KARAKTERISTIK DIAMETER GRAF (SLIDE 24) ===")
print(f"• Diameter Graf (d_maks)              : {diameter_val}")
print(f"• Jarak Rata-rata (Average Path Length) : {avg_path_length:.4f}")
print(f"• Lintasan Geodesik Terpanjang          : {' -> '.join(map(str, lintasan_terpanjang))}")
print(f"• Panjang Sisi Lintasan                 : {len(lintasan_terpanjang) - 1}")

# Visualisasi Graf dengan Highlight Lintasan Diameter
plt.figure(figsize=(10, 6))
pos_d = nx.spring_layout(G_slide24, seed=15)

# Gambar seluruh sisi
nx.draw_networkx_edges(G_slide24, pos_d, width=1.5, edge_color='lightgray')

# Highlight sisi lintasan diameter dengan warna merah tebal
path_edges = list(zip(lintasan_terpanjang[:-1], lintasan_terpanjang[1:]))
nx.draw_networkx_edges(G_slide24, pos_d, edgelist=path_edges, width=3.5, edge_color='red')

# Gambar simpul
node_colors = ['gold' if n in lintasan_terpanjang else 'lightblue' for n in G_slide24.nodes()]
nx.draw_networkx_nodes(G_slide24, pos_d, node_color=node_colors, node_size=900, edgecolors='black')
nx.draw_networkx_labels(G_slide24, pos_d, font_size=11, font_weight='bold')

plt.title(f"Visualisasi Graf Slide 24: Diameter = {diameter_val}\nLintasan Terpanjang: 7 - 2 - 6 - 3 - 4 - 5 (Merah)", 
          fontsize=12, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 11. Rangkuman Metrik Karakteristik Graf

Berikut adalah rangkuman seluruh rumus dan metrik karakteristik struktur graf dari Slide 1 hingga Slide 24:

| Karakteristik Graf | Graf Tak Berarah (*Undirected*) | Graf Berarah (*Directed*) | Fungsi NetworkX |
| :--- | :--- | :--- | :--- |
| **Derajat Simpul ($k_v$)** | $k_v = \text{deg}(v)$ | $k_v^{in} = \sum a_{ji}, \quad k_v^{out} = \sum a_{ij}$ | `G.degree()`, `G.in_degree()`, `G.out_degree()` |
| **Derajat Rata-rata ($\langle k \rangle$)** | $\langle k \rangle = \frac{2L}{N}$ | $\langle k^{in} \rangle = \langle k^{out} \rangle = \frac{L}{N}$ | `sum(dict(G.degree()).values()) / N` |
| **Sisi Maksimal ($L_{maks}$)** | $L_{maks} = \frac{N(N - 1)}{2}$ | $L_{maks} = N(N - 1)$ | Manual formula |
| **Kepadatan Graf (*Density*)** | $D = \frac{2L}{N(N - 1)}$ | $D = \frac{L}{N(N - 1)}$ | `nx.density(G)` |
| **Koefisien Klasterisasi ($C_i$)** | $C_i = \frac{2 k_i}{d_i(d_i - 1)}$ | Fagiolo directed clustering | `nx.clustering(G)` |
| **Klasterisasi Rata-rata ($\bar{C}$)** | $\bar{C} = \frac{1}{N} \sum C_i$ | $\bar{C} = \frac{1}{N} \sum C_i$ | `nx.average_clustering(G)` |
| **Jarak Geodesik ($d(u,v)$)** | $d(u, v) = d(v, u)$ | Umumnya $d(u, v) \neq d(v, u)$ | `nx.shortest_path_length(G, u, v)` |
| **Jarak Rata-rata ($\langle d \rangle$)** | $\langle d \rangle = \frac{1}{N(N-1)} \sum_{i \neq j} d_{ij}$ | $\langle d \rangle = \frac{1}{N(N-1)} \sum_{i \neq j} d_{ij}$ | `nx.average_shortest_path_length(G)` |
| **Diameter Jaringan ($d_{maks}$)** | $d_{maks} = \max d(u, v)$ | $d_{maks} = \max d(u, v)$ | `nx.diameter(G)` |